In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
train_B2000WCN_elasticnet_v2.py

Train interpretable ML models using B2000WCN only.

Training predictors:
    O3_Feb_mean
    O3_Feb_trend
    NAM50_Feb_mean
    Fz100_DJF_mean
    PSC197_DJF_Tmin50

Targets:
    Regression:
        MA_O3_loss = -mean(Mar-Apr O3 anomaly)

    Classification:
        Low25_min_label from low25pct_years_30_70hPa.txt

Important:
    Use xarray .dt for cftime model years.
    Do NOT use pandas.to_datetime for years like 0001.
"""

from pathlib import Path
import re
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import xarray as xr

from scipy.stats import pearsonr

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV, LogisticRegressionCV
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    mean_squared_error,
    roc_auc_score,
    brier_score_loss,
    accuracy_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")


# ============================================================
# Paths and configuration
# ============================================================

EXP_NAME = "B2000WCN"

BASE_PROCESSED = Path(
    "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
)

O3_FILE = BASE_PROCESSED / "B2000WCN" / "O3_pc_B2000WCN_30_70hPa.nc"
TMIN_FILE = BASE_PROCESSED / "B2000WCN" / "T_min_60_90N_B2000WCN.nc"
LOW25_FILE = BASE_PROCESSED / "B2000WCN" / "low25pct_years_30_70hPa.txt"

NAM_FILE = Path(
    "/mnt/soclim0/public_data/weiji/B2000WCN001002/NAM/B2000WCN001002_Vertical_NAM.nc"
)

EP_FILE = Path(
    "/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"
)

OUT_DIR = Path("/home/weiji/restart_exam/code/20260426ML/ozone_ml/outputs/B2000WCN_training_v2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = OUT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_CSV = OUT_DIR / "B2000WCN_annual_ML_features.csv"
SUMMARY_JSON = OUT_DIR / "B2000WCN_training_skill_summary.json"

EXCLUDE_YEARS = {104}
TCRIT = 197.0
RANDOM_STATE = 42


# ============================================================
# Generic helpers
# ============================================================

def open_dataarray_or_first_var(path):
    """
    Open as DataArray if possible.
    If file is Dataset, return first data variable.
    """
    try:
        return xr.open_dataarray(path, use_cftime=True)
    except Exception:
        ds = xr.open_dataset(path, use_cftime=True)
        var = list(ds.data_vars)[0]
        return ds[var]


def detect_data_var(ds, candidates):
    for c in candidates:
        if c in ds.data_vars:
            return c
    lower = {v.lower(): v for v in ds.data_vars}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    if len(ds.data_vars) == 1:
        return list(ds.data_vars)[0]
    raise ValueError(f"Cannot detect variable. Available: {list(ds.data_vars)}")


def detect_coord(da, candidates):
    names = list(da.coords) + list(da.dims)
    for c in candidates:
        if c in names:
            return c
    raise ValueError(f"Cannot detect coordinate from {candidates}. Available: {names}")


def select_pressure(da, target_hpa):
    """
    Select pressure coordinate robustly.
    If max pressure > 2000, assume Pa.
    Otherwise assume hPa.
    """
    lev_name = detect_coord(da, ["lev", "plev", "level", "pressure"])
    lev_vals = np.asarray(da[lev_name].values, dtype=float)

    if np.nanmax(lev_vals) > 2000:
        target = target_hpa * 100.0
    else:
        target = target_hpa

    return da.sel({lev_name: target}, method="nearest")


def da_to_df_with_time(da, value_name):
    """
    Convert DataArray to dataframe and add cftime-safe time fields
    using xarray .dt, not pandas datetime.
    """
    da = da.rename(value_name)

    df = da.to_dataframe().reset_index()

    time_info = pd.DataFrame({
        "time": da["time"].values,
        "year": da["time"].dt.year.values.astype(int),
        "month": da["time"].dt.month.values.astype(int),
        "day": da["time"].dt.day.values.astype(int),
        "doy": da["time"].dt.dayofyear.values.astype(int),
    })

    df = df.merge(time_info, on="time", how="left")

    df = df[~((df["month"] == 2) & (df["day"] == 29))].copy()
    df = df[~df["year"].isin(EXCLUDE_YEARS)].copy()

    return df


def month_day_string(df):
    return df["month"].astype(str).str.zfill(2) + "-" + df["day"].astype(str).str.zfill(2)


def linear_trend(y, x):
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)

    if valid.sum() < 5:
        return np.nan

    return float(np.polyfit(x[valid], y[valid], 1)[0])


def parse_low25_years(path):
    txt = Path(path).read_text()
    nums = re.findall(r"\b\d+\b", txt)
    return sorted(set(int(n) for n in nums))


# ============================================================
# O3 features and target
# ============================================================

def build_o3_annual():
    print("\nReading O3:", O3_FILE)

    da = open_dataarray_or_first_var(O3_FILE)

    # 5-day centered running mean, consistent with your low-O3 diagnostic.
    da5 = da.rolling(time=5, center=True, min_periods=3).mean()

    df = da_to_df_with_time(da5, "o3_3070_5d")

    # Daily climatology by month-day
    df["month_day"] = month_day_string(df)
    clim = (
        df.groupby("month_day")["o3_3070_5d"]
        .mean()
        .rename("o3_clim")
        .reset_index()
    )

    df = df.merge(clim, on="month_day", how="left")
    df["o3_anom"] = df["o3_3070_5d"] - df["o3_clim"]

    rows = []

    for year, g in df.groupby("year"):
        g = g.sort_values("doy")

        feb = g[g["month"] == 2]
        ma = g[g["month"].isin([3, 4])]

        if len(feb) < 20 or len(ma) < 50:
            continue

        row = {"year": int(year)}

        row["O3_Feb_mean"] = float(feb["o3_anom"].mean())
        row["O3_Feb_trend"] = linear_trend(
            feb["o3_anom"].values,
            feb["doy"].values
        )

        row["MA_O3_anom"] = float(ma["o3_anom"].mean())
        row["MA_O3_loss"] = -row["MA_O3_anom"]
        row["MA_O3_min"] = float(ma["o3_3070_5d"].min())

        idx_min = ma["o3_3070_5d"].idxmin()
        row["MA_O3_min_doy"] = int(ma.loc[idx_min, "doy"])

        rows.append(row)

    out = pd.DataFrame(rows)
    print("O3 annual:", out.shape, out["year"].min(), out["year"].max())
    return out


# ============================================================
# T / PSC cold exposure
# ============================================================

def build_psc_annual():
    print("\nReading T_min:", TMIN_FILE)

    da = open_dataarray_or_first_var(TMIN_FILE)
    da50 = select_pressure(da, 50)

    df = da_to_df_with_time(da50, "Tmin50")

    df["season_year"] = df["year"]
    df.loc[df["month"] == 12, "season_year"] = (
        df.loc[df["month"] == 12, "year"] + 1
    )

    djf = df[df["month"].isin([12, 1, 2])].copy()
    djf["cold_deficit"] = np.maximum(TCRIT - djf["Tmin50"].astype(float), 0.0)

    rows = []

    for sy, g in djf.groupby("season_year"):
        sy = int(sy)

        if sy in EXCLUDE_YEARS:
            continue

        if len(g) < 75:
            continue

        rows.append({
            "year": sy,
            "PSC197_DJF_Tmin50": float(g["cold_deficit"].sum()),
            "PSC197_DJF_days_below": int((g["cold_deficit"] > 0).sum()),
            "Tmin50_DJF_min": float(g["Tmin50"].min()),
            "Tmin50_DJF_mean": float(g["Tmin50"].mean()),
            "n_days_DJF_T": int(len(g)),
        })

    out = pd.DataFrame(rows)
    print("PSC annual:", out.shape, out["year"].min(), out["year"].max())
    return out


# ============================================================
# NAM50
# ============================================================

def build_nam_annual():
    print("\nReading NAM:", NAM_FILE)

    ds = xr.open_dataset(NAM_FILE, use_cftime=True)
    var = detect_data_var(ds, ["NAM", "NAM_Vertical", "nam", "nam_vertical"])
    da = ds[var]

    da50 = select_pressure(da, 50)
    df = da_to_df_with_time(da50, "NAM50")

    rows = []

    for year, g in df.groupby("year"):
        feb = g[g["month"] == 2]

        if len(feb) < 20:
            continue

        rows.append({
            "year": int(year),
            "NAM50_Feb_mean": float(feb["NAM50"].mean()),
            "n_days_Feb_NAM": int(len(feb)),
        })

    out = pd.DataFrame(rows)
    print("NAM annual:", out.shape, out["year"].min(), out["year"].max())
    return out


# ============================================================
# EP flux
# ============================================================

def build_fz_annual():
    print("\nReading EP flux:", EP_FILE)

    ds = xr.open_dataset(EP_FILE, use_cftime=True)
    var = detect_data_var(ds, ["Fz", "EP2_cart", "ep2_cart", "FZ"])
    da = ds[var]

    da100 = select_pressure(da, 100)

    # This file has no latitude dimension according to debug output.
    # So we use the selected 100 hPa time series directly.
    df = da_to_df_with_time(da100, "Fz100")

    # B2000WCN001002 EP flux year correction.
    df["year_original"] = df["year"].astype(int)
    df.loc[df["year_original"] >= 105, "year"] = (
        df.loc[df["year_original"] >= 105, "year_original"] + 1
    )

    df = df[~df["year"].isin(EXCLUDE_YEARS)].copy()

    df["season_year"] = df["year"]
    df.loc[df["month"] == 12, "season_year"] = (
        df.loc[df["month"] == 12, "year"] + 1
    )

    djf = df[df["month"].isin([12, 1, 2])].copy()

    rows = []

    for sy, g in djf.groupby("season_year"):
        sy = int(sy)

        if sy in EXCLUDE_YEARS:
            continue

        if len(g) < 75:
            continue

        rows.append({
            "year": sy,
            "Fz100_DJF_mean": float(g["Fz100"].mean()),
            "n_days_DJF_Fz": int(len(g)),
            "Fz_orig_year_min": int(g["year_original"].min()),
            "Fz_orig_year_max": int(g["year_original"].max()),
        })

    out = pd.DataFrame(rows)
    print("Fz annual:", out.shape, out["year"].min(), out["year"].max())
    return out


# ============================================================
# Build merged feature table
# ============================================================

def build_feature_table():
    o3 = build_o3_annual()
    psc = build_psc_annual()
    nam = build_nam_annual()
    fz = build_fz_annual()

    df = o3.copy()
    for tmp in [psc, nam, fz]:
        df = df.merge(tmp, on="year", how="inner")

    low25_years = parse_low25_years(LOW25_FILE)
    df["Low25_min_label"] = df["year"].isin(low25_years).astype(int)

    loss_threshold = df["MA_O3_loss"].quantile(0.75)
    df["Low25_loss_label"] = (df["MA_O3_loss"] >= loss_threshold).astype(int)
    df["Low25_loss_threshold"] = loss_threshold

    df["exp"] = EXP_NAME

    df = df.sort_values("year").reset_index(drop=True)

    print("\nFinal merged feature table:")
    print(df.head())
    print(df.tail())
    print("shape:", df.shape)
    print("year min/max:", df["year"].min(), df["year"].max())
    print("Low25_min count:", int(df["Low25_min_label"].sum()))
    print("Low25_loss count:", int(df["Low25_loss_label"].sum()))

    df.to_csv(FEATURE_CSV, index=False)
    print("\nSaved feature table:", FEATURE_CSV)

    return df


# ============================================================
# Model training
# ============================================================

PREDICTOR_SETS = {
    "DYN": [
        "NAM50_Feb_mean",
        "Fz100_DJF_mean",
        "PSC197_DJF_Tmin50",
    ],
    "O3": [
        "O3_Feb_mean",
        "O3_Feb_trend",
    ],
    "ALL": [
        "O3_Feb_mean",
        "O3_Feb_trend",
        "NAM50_Feb_mean",
        "Fz100_DJF_mean",
        "PSC197_DJF_Tmin50",
    ],
}


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))

    y_clim = np.full_like(y_true, y_true.mean())
    mse_clim = mean_squared_error(y_true, y_clim)

    skill = float(1.0 - mse / mse_clim) if mse_clim > 0 else np.nan

    r, p = pearsonr(y_true, y_pred)

    return {
        "rmse": rmse,
        "mse_skill_vs_climatology": skill,
        "correlation": float(r),
        "correlation_pvalue": float(p),
    }


def classification_metrics(y_true, prob):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob).astype(float)
    pred = (prob >= 0.5).astype(int)

    out = {
        "roc_auc": float(roc_auc_score(y_true, prob)),
        "brier_score": float(brier_score_loss(y_true, prob)),
        "accuracy_threshold_0p5": float(accuracy_score(y_true, pred)),
    }

    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    out["hit_rate_threshold_0p5"] = float(tp / (tp + fn)) if (tp + fn) > 0 else np.nan
    out["false_alarm_rate_threshold_0p5"] = float(fp / (fp + tn)) if (fp + tn) > 0 else np.nan

    return out


def make_regression_model(n_samples):
    cv_inner = min(5, max(3, n_samples // 20))

    return Pipeline([
        ("scaler", StandardScaler()),
        ("reg", ElasticNetCV(
            l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
            alphas=np.logspace(-4, 2, 100),
            cv=cv_inner,
            max_iter=50000,
            random_state=RANDOM_STATE,
        ))
    ])


def make_logistic_model(y):
    counts = np.bincount(np.asarray(y).astype(int))
    cv_inner = min(5, counts.min())

    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegressionCV(
            Cs=np.logspace(-3, 3, 40),
            penalty="elasticnet",
            solver="saga",
            l1_ratios=[0.1, 0.3, 0.5, 0.7, 0.9],
            cv=cv_inner,
            scoring="roc_auc",
            class_weight="balanced",
            max_iter=50000,
            random_state=RANDOM_STATE,
        ))
    ])


def extract_coefficients(model, features, kind):
    if kind == "regression":
        m = model.named_steps["reg"]
        coef = m.coef_
        intercept = m.intercept_
    elif kind == "classification":
        m = model.named_steps["clf"]
        coef = m.coef_.ravel()
        intercept = m.intercept_[0]
    else:
        raise ValueError(kind)

    return {
        "intercept": float(intercept),
        "coefficients_standardized": {
            f: float(c) for f, c in zip(features, coef)
        }
    }


def train_models(df):
    target_reg = "MA_O3_loss"
    target_cls = "Low25_min_label"

    summary = {
        "training_dataset": EXP_NAME,
        "target_regression": target_reg,
        "target_classification": target_cls,
        "predictor_sets": PREDICTOR_SETS,
        "models": {},
    }

    for name, features in PREDICTOR_SETS.items():
        print("\n" + "=" * 80)
        print("Training predictor set:", name)
        print(features)

        sub = df.dropna(subset=features + [target_reg, target_cls]).copy()

        X = sub[features].values
        y_reg = sub[target_reg].values.astype(float)
        y_cls = sub[target_cls].values.astype(int)
        years = sub["year"].values.astype(int)

        print("N samples:", len(sub))
        print("N low25:", int(y_cls.sum()))

        # -------------------------
        # Regression CV
        # -------------------------
        cv_reg = KFold(
            n_splits=min(5, len(sub)),
            shuffle=True,
            random_state=RANDOM_STATE,
        )

        reg_cv_model = make_regression_model(len(sub))

        y_reg_pred_cv = cross_val_predict(
            reg_cv_model,
            X,
            y_reg,
            cv=cv_reg,
            method="predict",
        )

        reg_skill = regression_metrics(y_reg, y_reg_pred_cv)

        reg_final = make_regression_model(len(sub))
        reg_final.fit(X, y_reg)
        reg_coef = extract_coefficients(reg_final, features, "regression")

        reg_file = MODEL_DIR / f"elasticnet_regression_{name}.joblib"
        joblib.dump({
            "model": reg_final,
            "features": features,
            "target": target_reg,
            "training_dataset": EXP_NAME,
            "training_years": years,
            "coefficients": reg_coef,
        }, reg_file)

        # -------------------------
        # Classification CV
        # -------------------------
        cv_cls = StratifiedKFold(
            n_splits=min(5, np.bincount(y_cls).min()),
            shuffle=True,
            random_state=RANDOM_STATE,
        )

        clf_cv_model = make_logistic_model(y_cls)

        prob_cv = cross_val_predict(
            clf_cv_model,
            X,
            y_cls,
            cv=cv_cls,
            method="predict_proba",
        )[:, 1]

        cls_skill = classification_metrics(y_cls, prob_cv)

        clf_final = make_logistic_model(y_cls)
        clf_final.fit(X, y_cls)
        clf_coef = extract_coefficients(clf_final, features, "classification")

        clf_file = MODEL_DIR / f"elasticnet_logistic_{name}.joblib"
        joblib.dump({
            "model": clf_final,
            "features": features,
            "target": target_cls,
            "training_dataset": EXP_NAME,
            "training_years": years,
            "coefficients": clf_coef,
        }, clf_file)

        # Save CV predictions
        pred_df = pd.DataFrame({
            "year": years,
            "MA_O3_loss_true": y_reg,
            f"MA_O3_loss_pred_{name}": y_reg_pred_cv,
            "Low25_true": y_cls,
            f"P_lowO3_{name}": prob_cv,
        })

        pred_file = OUT_DIR / f"cv_predictions_{name}.csv"
        pred_df.to_csv(pred_file, index=False)

        summary["models"][name] = {
            "n_samples": int(len(sub)),
            "n_low25": int(y_cls.sum()),
            "features": features,
            "regression_metrics": reg_skill,
            "classification_metrics": cls_skill,
            "regression_model_file": str(reg_file),
            "classification_model_file": str(clf_file),
            "cv_predictions_file": str(pred_file),
            "regression_coefficients": reg_coef,
            "classification_coefficients": clf_coef,
        }

        print("\nRegression skill:")
        print(json.dumps(reg_skill, indent=2))

        print("\nClassification skill:")
        print(json.dumps(cls_skill, indent=2))

        print("\nRegression coefficients:")
        print(json.dumps(reg_coef, indent=2))

        print("\nClassification coefficients:")
        print(json.dumps(clf_coef, indent=2))

    with open(SUMMARY_JSON, "w") as f:
        json.dump(summary, f, indent=2)

    print("\nSaved summary:", SUMMARY_JSON)


# ============================================================
# Main
# ============================================================

def main():
    print("=" * 80)
    print("B2000WCN ML training v2")
    print("=" * 80)

    print("O3_FILE:", O3_FILE)
    print("TMIN_FILE:", TMIN_FILE)
    print("NAM_FILE:", NAM_FILE)
    print("EP_FILE:", EP_FILE)
    print("LOW25_FILE:", LOW25_FILE)

    df = build_feature_table()
    train_models(df)

    print("\nDone.")


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
test_on_external_sets_v1.py

Apply trained B2000WCN models to external test datasets:
    1) B2000WCN_NOCOUPL
    2) BWCN
    3) REANALYSIS

Models are loaded from:
    /home/weiji/restart_exam/code/20260426ML/ozone_ml/outputs/B2000WCN_training_v2/models

This script:
    - builds annual feature tables for each test dataset
    - applies trained regression/classification models
    - computes evaluation metrics
    - saves prediction tables
    - generates diagnostic figures

Author: ChatGPT + Jim project workflow
"""

from pathlib import Path
import re
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from scipy.stats import pearsonr
from sklearn.metrics import (
    mean_squared_error,
    roc_auc_score,
    brier_score_loss,
    accuracy_score,
    confusion_matrix,
    roc_curve,
)

warnings.filterwarnings("ignore")


# ============================================================
# Paths
# ============================================================

BASE_PROCESSED = Path(
    "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
)

TRAIN_ROOT = Path(
    "/home/weiji/restart_exam/code/20260426ML/ozone_ml/outputs/B2000WCN_training_v2"
)

MODEL_DIR = TRAIN_ROOT / "models"

OUT_ROOT = TRAIN_ROOT / "test_evaluation_v1"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

SUMMARY_JSON = OUT_ROOT / "external_test_summary.json"
SUMMARY_CSV = OUT_ROOT / "external_test_summary.csv"

# Reanalysis NAM / EP paths
REANALYSIS_NAM = Path(
    "/home/weiji/restart_exam/code/20260315NAM/resul/New_NAM_Algorithm/ERA5_daily_vertical_NAM_monthlyEOF_DOYprojected.nc"
)

REANALYSIS_EP = Path(
    "/mnt/soclim0/public_data/weiji/MERRA2_Processed/EPflux_daily/MERRA2_EPFLUX_Daily_Series_1980_2026_noW.nc"
)

BWCN_EP = Path(
    "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
)

BWCN_NAM = Path(
    "/mnt/soclim0/public_data/weiji/BWCN/NAM/BWCN_Vertical_NAM.nc"
)

NOCOUPL_NAM = Path(
    "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/NAM/B2000WCN_NOCOUPL001002_Vertical_NAM.nc"
)

NOCOUPL_EP = Path(
    "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"
)

# ============================================================
# Configuration
# ============================================================

TCRIT = 197.0
RANDOM_STATE = 42

PREDICTOR_SETS = {
    "DYN": [
        "NAM50_Feb_mean",
        "Fz100_DJF_mean",
        "PSC197_DJF_Tmin50",
    ],
    "O3": [
        "O3_Feb_mean",
        "O3_Feb_trend",
    ],
    "ALL": [
        "O3_Feb_mean",
        "O3_Feb_trend",
        "NAM50_Feb_mean",
        "Fz100_DJF_mean",
        "PSC197_DJF_Tmin50",
    ],
}

TEST_DATASETS = {
    "B2000WCN_NOCOUPL": {
        "o3_file": BASE_PROCESSED / "B2000WCN_NOCOUPL" / "O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc",
        "tmin_file": BASE_PROCESSED / "B2000WCN_NOCOUPL" / "T_min_60_90N_B2000WCN_NOCOUPL.nc",
        "low25_file": BASE_PROCESSED / "B2000WCN_NOCOUPL" / "low25pct_years_30_70hPa.txt",
        "nam_file": NOCOUPL_NAM,
        "ep_file": NOCOUPL_EP,
        "exclude_years": {104},
        "ep_needs_year_correction": True,
    },
    "BWCN": {
        "o3_file": BASE_PROCESSED / "BWCN" / "O3_pc_BWCN_30_70hPa.nc",
        "tmin_file": BASE_PROCESSED / "BWCN" / "T_min_60_90N_BWCN.nc",
        "low25_file": BASE_PROCESSED / "BWCN" / "low25pct_years_30_70hPa.txt",
        "nam_file": BWCN_NAM,
        "ep_file": BWCN_EP,
        "exclude_years": set(),
        "ep_needs_year_correction": False,
    },
    "REANALYSIS": {
        "o3_file": BASE_PROCESSED / "MERRA2" / "O3_pc_MERRA2_30_70hPa.nc",
        "tmin_file": None,   # auto-find below
        "low25_file": BASE_PROCESSED / "MERRA2" / "low25pct_years_30_70hPa.txt",
        "nam_file": REANALYSIS_NAM,
        "ep_file": REANALYSIS_EP,
        "exclude_years": set(),
        "ep_needs_year_correction": False,
    },
}


# ============================================================
# Helpers
# ============================================================

def auto_find_merra2_tmin():
    merra2_dir = BASE_PROCESSED / "MERRA2"
    patterns = [
        "T_min_60_90N*.nc",
        "*T_min*60_90N*.nc",
        "*Tmin*60_90N*.nc",
        "*.nc",
    ]
    for pat in patterns:
        files = sorted(merra2_dir.glob(pat))
        for f in files:
            name = f.name.lower()
            if ("t" in name) and ("min" in name):
                return f
    raise FileNotFoundError(f"Cannot find MERRA2 T_min file in {merra2_dir}")


def open_dataarray_or_first_var(path):
    try:
        return xr.open_dataarray(path, use_cftime=True)
    except Exception:
        ds = xr.open_dataset(path, use_cftime=True)
        var = list(ds.data_vars)[0]
        return ds[var]


def detect_data_var(ds, candidates):
    for c in candidates:
        if c in ds.data_vars:
            return c
    lower = {v.lower(): v for v in ds.data_vars}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    if len(ds.data_vars) == 1:
        return list(ds.data_vars)[0]
    raise ValueError(f"Cannot detect variable. Available: {list(ds.data_vars)}")


def detect_coord(da, candidates):
    names = list(da.coords) + list(da.dims)
    for c in candidates:
        if c in names:
            return c
    return None


def select_pressure(da, target_hpa):
    lev_name = detect_coord(da, ["lev", "plev", "level", "pressure", "p"])
    if lev_name is None:
        raise ValueError(f"No pressure coordinate found in {da.dims} / {list(da.coords)}")

    lev_vals = np.asarray(da[lev_name].values, dtype=float)
    if np.nanmax(lev_vals) > 2000:
        target = target_hpa * 100.0
    else:
        target = target_hpa

    return da.sel({lev_name: target}, method="nearest")


def da_to_df_with_time(da, value_name, exclude_years=None):
    if exclude_years is None:
        exclude_years = set()

    da = da.rename(value_name)
    df = da.to_dataframe().reset_index()

    time_info = pd.DataFrame({
        "time": da["time"].values,
        "year": da["time"].dt.year.values.astype(int),
        "month": da["time"].dt.month.values.astype(int),
        "day": da["time"].dt.day.values.astype(int),
        "doy": da["time"].dt.dayofyear.values.astype(int),
    })

    df = df.merge(time_info, on="time", how="left")

    df = df[~((df["month"] == 2) & (df["day"] == 29))].copy()
    if len(exclude_years) > 0:
        df = df[~df["year"].isin(exclude_years)].copy()

    return df


def month_day_string(df):
    return df["month"].astype(str).str.zfill(2) + "-" + df["day"].astype(str).str.zfill(2)


def linear_trend(y, x):
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 5:
        return np.nan
    return float(np.polyfit(x[valid], y[valid], 1)[0])


def parse_low25_years(path):
    txt = Path(path).read_text()
    nums = re.findall(r"\b\d+\b", txt)
    return sorted(set(int(n) for n in nums))


# ============================================================
# Annual feature building
# ============================================================

def build_o3_annual(o3_file, exclude_years):
    da = open_dataarray_or_first_var(o3_file)

    da5 = da.rolling(time=5, center=True, min_periods=3).mean()
    df = da_to_df_with_time(da5, "o3_3070_5d", exclude_years=exclude_years)

    df["month_day"] = month_day_string(df)
    clim = (
        df.groupby("month_day")["o3_3070_5d"]
        .mean()
        .rename("o3_clim")
        .reset_index()
    )

    df = df.merge(clim, on="month_day", how="left")
    df["o3_anom"] = df["o3_3070_5d"] - df["o3_clim"]

    rows = []
    for year, g in df.groupby("year"):
        g = g.sort_values("doy")

        feb = g[g["month"] == 2]
        ma = g[g["month"].isin([3, 4])]

        if len(feb) < 20 or len(ma) < 50:
            continue

        row = {"year": int(year)}
        row["O3_Feb_mean"] = float(feb["o3_anom"].mean())
        row["O3_Feb_trend"] = linear_trend(feb["o3_anom"].values, feb["doy"].values)
        row["MA_O3_anom"] = float(ma["o3_anom"].mean())
        row["MA_O3_loss"] = -row["MA_O3_anom"]
        row["MA_O3_min"] = float(ma["o3_3070_5d"].min())

        idx_min = ma["o3_3070_5d"].idxmin()
        row["MA_O3_min_doy"] = int(ma.loc[idx_min, "doy"])

        rows.append(row)

    return pd.DataFrame(rows)


def build_psc_annual(tmin_file, exclude_years):
    da = open_dataarray_or_first_var(tmin_file)
    da50 = select_pressure(da, 50)

    df = da_to_df_with_time(da50, "Tmin50", exclude_years=exclude_years)

    df["season_year"] = df["year"]
    df.loc[df["month"] == 12, "season_year"] = df.loc[df["month"] == 12, "year"] + 1

    djf = df[df["month"].isin([12, 1, 2])].copy()
    djf["cold_deficit"] = np.maximum(TCRIT - djf["Tmin50"].astype(float), 0.0)

    rows = []
    for sy, g in djf.groupby("season_year"):
        sy = int(sy)
        if sy in exclude_years:
            continue
        if len(g) < 75:
            continue

        rows.append({
            "year": sy,
            "PSC197_DJF_Tmin50": float(g["cold_deficit"].sum()),
            "PSC197_DJF_days_below": int((g["cold_deficit"] > 0).sum()),
            "Tmin50_DJF_min": float(g["Tmin50"].min()),
            "Tmin50_DJF_mean": float(g["Tmin50"].mean()),
            "n_days_DJF_T": int(len(g)),
        })

    return pd.DataFrame(rows)


def build_nam_annual(nam_file, exclude_years):
    ds = xr.open_dataset(nam_file, use_cftime=True)
    var = detect_data_var(ds, ["NAM", "NAM_Vertical", "nam", "nam_vertical"])
    da = ds[var]

    da50 = select_pressure(da, 50)
    df = da_to_df_with_time(da50, "NAM50", exclude_years=exclude_years)

    rows = []
    for year, g in df.groupby("year"):
        feb = g[g["month"] == 2]
        if len(feb) < 20:
            continue

        rows.append({
            "year": int(year),
            "NAM50_Feb_mean": float(feb["NAM50"].mean()),
            "n_days_Feb_NAM": int(len(feb)),
        })

    return pd.DataFrame(rows)


def build_fz_annual(ep_file, exclude_years, ep_needs_year_correction=False):
    ds = xr.open_dataset(ep_file, use_cftime=True)
    var = detect_data_var(ds, ["Fz", "EP2_cart", "ep2_cart", "FZ"])
    da = ds[var]

    da100 = select_pressure(da, 100)

    lat_name = detect_coord(da100, ["lat", "latitude", "LAT", "Latitude", "lats"])
    if lat_name is not None:
        try:
            sub = da100.sel({lat_name: slice(40, 80)})
            if sub.sizes.get(lat_name, 0) == 0:
                sub = da100.sel({lat_name: slice(80, 40)})
            weights = np.cos(np.deg2rad(sub[lat_name]))
            da100 = sub.weighted(weights).mean(lat_name)
        except Exception:
            pass

    df = da_to_df_with_time(da100, "Fz100", exclude_years=exclude_years)

    if ep_needs_year_correction:
        df["year_original"] = df["year"].astype(int)
        df.loc[df["year_original"] >= 105, "year"] = df.loc[df["year_original"] >= 105, "year_original"] + 1
        df = df[~df["year"].isin(exclude_years)].copy()
    else:
        df["year_original"] = df["year"].astype(int)

    df["season_year"] = df["year"]
    df.loc[df["month"] == 12, "season_year"] = df.loc[df["month"] == 12, "year"] + 1

    djf = df[df["month"].isin([12, 1, 2])].copy()

    rows = []
    for sy, g in djf.groupby("season_year"):
        sy = int(sy)
        if sy in exclude_years:
            continue
        if len(g) < 75:
            continue

        rows.append({
            "year": sy,
            "Fz100_DJF_mean": float(g["Fz100"].mean()),
            "n_days_DJF_Fz": int(len(g)),
            "Fz_orig_year_min": int(g["year_original"].min()),
            "Fz_orig_year_max": int(g["year_original"].max()),
        })

    return pd.DataFrame(rows)


def build_feature_table_for_dataset(ds_name, cfg):
    exclude_years = cfg["exclude_years"]

    tmin_file = cfg["tmin_file"]
    if tmin_file is None and ds_name == "REANALYSIS":
        tmin_file = auto_find_merra2_tmin()

    print("\n" + "=" * 80)
    print(f"Building feature table for {ds_name}")
    print("=" * 80)
    print("O3 :", cfg["o3_file"])
    print("T  :", tmin_file)
    print("NAM:", cfg["nam_file"])
    print("EP :", cfg["ep_file"])
    print("LOW25:", cfg["low25_file"])

    o3 = build_o3_annual(cfg["o3_file"], exclude_years=exclude_years)
    psc = build_psc_annual(tmin_file, exclude_years=exclude_years)
    nam = build_nam_annual(cfg["nam_file"], exclude_years=exclude_years)
    fz = build_fz_annual(
        cfg["ep_file"],
        exclude_years=exclude_years,
        ep_needs_year_correction=cfg["ep_needs_year_correction"]
    )

    df = o3.copy()
    for tmp in [psc, nam, fz]:
        df = df.merge(tmp, on="year", how="inner")

    low25_years = parse_low25_years(cfg["low25_file"])
    df["Low25_min_label"] = df["year"].isin(low25_years).astype(int)

    loss_threshold = df["MA_O3_loss"].quantile(0.75)
    df["Low25_loss_label"] = (df["MA_O3_loss"] >= loss_threshold).astype(int)
    df["Low25_loss_threshold"] = float(loss_threshold)

    df["exp"] = ds_name
    df = df.sort_values("year").reset_index(drop=True)

    print(df.head())
    print(df.tail())
    print("shape:", df.shape)
    print("year min/max:", df["year"].min(), df["year"].max())
    print("Low25 count:", int(df["Low25_min_label"].sum()))

    ds_out = OUT_ROOT / ds_name
    ds_out.mkdir(parents=True, exist_ok=True)
    csv_file = ds_out / f"{ds_name}_annual_features.csv"
    df.to_csv(csv_file, index=False)
    print("Saved feature table:", csv_file)

    return df


# ============================================================
# Metrics
# ============================================================

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))

    y_clim = np.full_like(y_true, y_true.mean())
    mse_clim = mean_squared_error(y_true, y_clim)
    skill = float(1.0 - mse / mse_clim) if mse_clim > 0 else np.nan

    try:
        r, p = pearsonr(y_true, y_pred)
    except Exception:
        r, p = np.nan, np.nan

    return {
        "rmse": rmse,
        "mse_skill_vs_climatology": skill,
        "correlation": float(r) if np.isfinite(r) else np.nan,
        "correlation_pvalue": float(p) if np.isfinite(p) else np.nan,
    }


def classification_metrics(y_true, prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob).astype(float)
    pred = (prob >= threshold).astype(int)

    out = {
        "brier_score": float(brier_score_loss(y_true, prob)),
        "accuracy_threshold_0p5": float(accuracy_score(y_true, pred)),
    }

    if len(np.unique(y_true)) == 2:
        out["roc_auc"] = float(roc_auc_score(y_true, prob))
    else:
        out["roc_auc"] = np.nan

    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    out["hit_rate_threshold_0p5"] = float(tp / (tp + fn)) if (tp + fn) > 0 else np.nan
    out["false_alarm_rate_threshold_0p5"] = float(fp / (fp + tn)) if (fp + tn) > 0 else np.nan
    out["tp"] = int(tp)
    out["fp"] = int(fp)
    out["tn"] = int(tn)
    out["fn"] = int(fn)

    return out


# ============================================================
# Plotting
# ============================================================

def plot_scatter_regression(ds_name, pred_df, out_dir):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
    sets = ["DYN", "O3", "ALL"]

    y_true = pred_df["MA_O3_loss_true"].values
    lo = np.nanmin(y_true)
    hi = np.nanmax(y_true)

    for ax, s in zip(axes, sets):
        y_pred = pred_df[f"MA_O3_loss_pred_{s}"].values

        ax.scatter(y_true, y_pred, alpha=0.8)
        ax.plot([lo, hi], [lo, hi], linestyle="--")
        ax.set_title(f"{ds_name} - {s}")
        ax.set_xlabel("True MA_O3_loss")
        ax.set_ylabel("Predicted MA_O3_loss")

        try:
            r, _ = pearsonr(y_true, y_pred)
            rmse = np.sqrt(mean_squared_error(y_true, y_pred))
            ax.text(
                0.05, 0.95,
                f"r = {r:.2f}\nRMSE = {rmse:.2f}",
                transform=ax.transAxes,
                va="top",
                ha="left",
                bbox=dict(facecolor="white", alpha=0.8, edgecolor="0.7")
            )
        except Exception:
            pass

    plt.tight_layout()
    outfile = out_dir / f"{ds_name}_regression_scatter.png"
    plt.savefig(outfile, dpi=200, bbox_inches="tight")
    plt.close()
    print("Saved:", outfile)


def plot_timeseries_regression(ds_name, pred_df, out_dir):
    fig, ax = plt.subplots(figsize=(12, 5))

    years = pred_df["year"].values
    ax.plot(years, pred_df["MA_O3_loss_true"].values, label="True", linewidth=2.2)
    ax.plot(years, pred_df["MA_O3_loss_pred_DYN"].values, label="Pred DYN")
    ax.plot(years, pred_df["MA_O3_loss_pred_O3"].values, label="Pred O3")
    ax.plot(years, pred_df["MA_O3_loss_pred_ALL"].values, label="Pred ALL")

    low_years = pred_df.loc[pred_df["Low25_true"] == 1, "year"].values
    for yy in low_years:
        ax.axvline(yy, alpha=0.15)

    ax.set_title(f"{ds_name} - MA_O3_loss Time Series")
    ax.set_xlabel("Year")
    ax.set_ylabel("MA_O3_loss")
    ax.legend()
    plt.tight_layout()

    outfile = out_dir / f"{ds_name}_regression_timeseries.png"
    plt.savefig(outfile, dpi=200, bbox_inches="tight")
    plt.close()
    print("Saved:", outfile)


def plot_probability_timeseries(ds_name, pred_df, out_dir):
    fig, ax = plt.subplots(figsize=(12, 5))

    years = pred_df["year"].values
    ax.plot(years, pred_df["P_lowO3_DYN"].values, label="Prob DYN")
    ax.plot(years, pred_df["P_lowO3_O3"].values, label="Prob O3")
    ax.plot(years, pred_df["P_lowO3_ALL"].values, label="Prob ALL")

    low_df = pred_df[pred_df["Low25_true"] == 1]
    if len(low_df) > 0:
        ax.scatter(
            low_df["year"].values,
            np.full(len(low_df), 1.02),
            marker="*",
            s=80,
            label="Observed low25"
        )

    ax.axhline(0.5, linestyle="--")
    ax.set_ylim(-0.02, 1.08)
    ax.set_title(f"{ds_name} - Low O3 Probability")
    ax.set_xlabel("Year")
    ax.set_ylabel("Predicted probability")
    ax.legend()
    plt.tight_layout()

    outfile = out_dir / f"{ds_name}_probability_timeseries.png"
    plt.savefig(outfile, dpi=200, bbox_inches="tight")
    plt.close()
    print("Saved:", outfile)


def plot_roc_curves(ds_name, pred_df, out_dir):
    fig, ax = plt.subplots(figsize=(6, 6))
    y_true = pred_df["Low25_true"].values

    for s in ["DYN", "O3", "ALL"]:
        prob = pred_df[f"P_lowO3_{s}"].values
        if len(np.unique(y_true)) == 2:
            fpr, tpr, _ = roc_curve(y_true, prob)
            auc = roc_auc_score(y_true, prob)
            ax.plot(fpr, tpr, label=f"{s} (AUC={auc:.2f})")

    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"{ds_name} - ROC Curves")
    ax.legend()
    plt.tight_layout()

    outfile = out_dir / f"{ds_name}_roc_curves.png"
    plt.savefig(outfile, dpi=200, bbox_inches="tight")
    plt.close()
    print("Saved:", outfile)


# ============================================================
# Evaluation on one dataset
# ============================================================

def evaluate_one_dataset(ds_name, cfg):
    ds_out = OUT_ROOT / ds_name
    ds_out.mkdir(parents=True, exist_ok=True)

    df = build_feature_table_for_dataset(ds_name, cfg)

    pred_df = pd.DataFrame({
        "year": df["year"].values,
        "MA_O3_loss_true": df["MA_O3_loss"].values,
        "MA_O3_min_true": df["MA_O3_min"].values,
        "MA_O3_min_doy_true": df["MA_O3_min_doy"].values,
        "Low25_true": df["Low25_min_label"].values,
        "exp": ds_name,
    })

    ds_summary = {}

    for s, features in PREDICTOR_SETS.items():
        print("\n" + "-" * 80)
        print(f"{ds_name} | predictor set = {s}")
        print(features)

        X = df[features].values
        y_reg = df["MA_O3_loss"].values.astype(float)
        y_cls = df["Low25_min_label"].values.astype(int)

        reg_bundle = joblib.load(MODEL_DIR / f"elasticnet_regression_{s}.joblib")
        clf_bundle = joblib.load(MODEL_DIR / f"elasticnet_logistic_{s}.joblib")

        reg_model = reg_bundle["model"]
        clf_model = clf_bundle["model"]

        y_reg_pred = reg_model.predict(X)
        prob = clf_model.predict_proba(X)[:, 1]

        pred_df[f"MA_O3_loss_pred_{s}"] = y_reg_pred
        pred_df[f"P_lowO3_{s}"] = prob
        pred_df[f"Low25_pred_{s}_0p5"] = (prob >= 0.5).astype(int)

        reg_skill = regression_metrics(y_reg, y_reg_pred)
        cls_skill = classification_metrics(y_cls, prob, threshold=0.5)

        ds_summary[s] = {
            "n_samples": int(len(df)),
            "n_low25": int(y_cls.sum()),
            "features": features,
            "regression_metrics": reg_skill,
            "classification_metrics": cls_skill,
        }

        print("Regression skill:")
        print(json.dumps(reg_skill, indent=2))

        print("Classification skill:")
        print(json.dumps(cls_skill, indent=2))

    pred_file = ds_out / f"{ds_name}_predictions.csv"
    pred_df.to_csv(pred_file, index=False)
    print("\nSaved predictions:", pred_file)

    metrics_file = ds_out / f"{ds_name}_metrics.json"
    with open(metrics_file, "w") as f:
        json.dump(ds_summary, f, indent=2)
    print("Saved metrics:", metrics_file)

    # plots
    plot_scatter_regression(ds_name, pred_df, ds_out)
    plot_timeseries_regression(ds_name, pred_df, ds_out)
    plot_probability_timeseries(ds_name, pred_df, ds_out)
    plot_roc_curves(ds_name, pred_df, ds_out)

    return ds_summary


# ============================================================
# Main
# ============================================================

def main():
    print("=" * 100)
    print("External test evaluation for B2000WCN-trained models")
    print("=" * 100)
    print("TRAIN_ROOT:", TRAIN_ROOT)
    print("MODEL_DIR :", MODEL_DIR)
    print("OUT_ROOT  :", OUT_ROOT)

    if not MODEL_DIR.exists():
        raise FileNotFoundError(f"Model directory not found: {MODEL_DIR}")

    # Fill reanalysis T_min automatically
    if TEST_DATASETS["REANALYSIS"]["tmin_file"] is None:
        TEST_DATASETS["REANALYSIS"]["tmin_file"] = auto_find_merra2_tmin()

    all_summary = {}
    rows_for_csv = []

    for ds_name, cfg in TEST_DATASETS.items():
        summary = evaluate_one_dataset(ds_name, cfg)
        all_summary[ds_name] = summary

        for s in ["DYN", "O3", "ALL"]:
            row = {
                "dataset": ds_name,
                "predictor_set": s,
            }
            row.update({f"reg_{k}": v for k, v in summary[s]["regression_metrics"].items()})
            row.update({f"cls_{k}": v for k, v in summary[s]["classification_metrics"].items()})
            row["n_samples"] = summary[s]["n_samples"]
            row["n_low25"] = summary[s]["n_low25"]
            rows_for_csv.append(row)

    with open(SUMMARY_JSON, "w") as f:
        json.dump(all_summary, f, indent=2)
    print("\nSaved overall json summary:", SUMMARY_JSON)

    summary_df = pd.DataFrame(rows_for_csv)
    summary_df.to_csv(SUMMARY_CSV, index=False)
    print("Saved overall csv summary:", SUMMARY_CSV)

    print("\nDone.")


if __name__ == "__main__":
    main()